# Hi-RAG reproduction: Phase 3–5, Stage 2

1. Generate Stage 2 training queries with the LLM (Phase 3)
2. Train the Stage 2 re-ranker (Phase 4)
3. Run full Hi-RAG end to end with the LLM (Phase 5)

Runtime → Change runtime type → **T4 GPU**. Run the cells in order.

In [ ]:
!nvidia-smi -L

## 1. Code + install
Upload the latest `hirag-repro.zip`.

In [ ]:
%cd /content
from google.colab import files
up = files.upload()                      # choose hirag-repro.zip
name = list(up.keys())[0]                # works even if Colab renames it 'hirag-repro (1).zip'
!rm -rf /content/hirag-repro
!unzip -q -o "/content/{name}" -d /content/
%cd /content/hirag-repro
!pip install -q -r requirements.txt

## 2. Google Drive (keeps the data, the model and the LLM answers safe if Colab disconnects)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/hirag_stage2'
GEN_CACHE = '/content/drive/MyDrive/hirag_gen_cache.jsonl'
CACHE = '/content/drive/MyDrive/hirag_llm_cache.jsonl'
import os; os.makedirs(OUT, exist_ok=True)

## 3. API key (typed in, never saved)

In [ ]:
import os, getpass
os.environ['LLM_API_KEY'] = getpass.getpass('API key (sk-or-... for OpenRouter, gsk_... for Groq): ')

## 4. Model settings
OpenRouter + Qwen3-32B (the paper's model) is the default. Using Groq instead? Comment out the first two lines and use the Groq lines.

In [ ]:
MODEL = 'qwen/qwen3-32b'                       # OpenRouter
EXTRA = '{"reasoning": {"enabled": false}}'
# MODEL = 'qwen/qwen3.8-27b'                   # Groq
# EXTRA = '{"reasoning_effort": "none", "reasoning_format": "hidden"}'
COMMON = (f'--model {MODEL} --cache {CACHE} --embed-model BAAI/bge-large-en --max-tokens 128 '
          f'--min-interval 1 --out-dir /content/drive/MyDrive/hirag_results_or')
print(MODEL, EXTRA)

## 5. Check the code (about 1 minute)
All tests should pass, including the Stage 2 training test. If one fails, send me the output.

In [ ]:
!python -m pytest -q tests

## 6. Phase 3: training data
### Quick test: 3 tools
You should see 3 example queries printed at the end. Send me the output if anything looks wrong.

In [ ]:
!python scripts/gen_train_data.py --model {MODEL} --extra-body '{EXTRA}' --limit-tools 3 --cache {GEN_CACHE} --out {OUT}/test_queries.json

### All 201 tools (about 10–20 minutes)
Answers are cached on Drive: if it stops, run the cell again and it continues.

In [ ]:
!python scripts/gen_train_data.py --model {MODEL} --extra-body '{EXTRA}' --cache {GEN_CACHE} --out {OUT}/train_queries.json

## 7. Phase 4: train Stage 2 (about 5–10 minutes on the T4)
Prints the loss and validation accuracy for 30 epochs, then a table: Stage 1 only vs Stage 1 + Stage 2.

In [ ]:
!python scripts/train_stage2.py --data {OUT}/train_queries.json --out {OUT}/stage2.pt

## 8. Phase 5: full Hi-RAG with the LLM (paper Table 2)
Run the 5-query test first.

In [ ]:
!python scripts/run_e2e.py --method hirag --split single --limit 5 --stage2 {OUT}/stage2.pt {COMMON} --extra-body '{EXTRA}'

In [ ]:
!python scripts/run_e2e.py --method hirag --split single --k 1 --stage2 {OUT}/stage2.pt {COMMON} --extra-body '{EXTRA}'
!python scripts/run_e2e.py --method hirag --split multi  --k 3 --stage2 {OUT}/stage2.pt {COMMON} --extra-body '{EXTRA}'

## 9. Download everything

In [ ]:
!cd /content/drive/MyDrive && zip -q -r /content/stage2_results.zip hirag_stage2 hirag_results_or
from google.colab import files
files.download('/content/stage2_results.zip')